<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Lab 03 — Compaction with a Preservation Gate

## Learning goals

1. Treat the context window as a **fixed budget** (RAM), with durable memory as disk.
2. Trigger compaction at ~**70–80%** of budget — never at 99% in a panic.
3. **Pin** load-bearing facts; summarize only the cold, evictable span.
4. Install a **preservation gate**: must-answer probes must still pass after compaction.
5. Practice a **pre-compaction flush** — externalize essentials *before* the lossy step.

## Theory you need

**Compaction** is recall-first lossy compression of the working tier: fold older turns into a synopsis, splice the synopsis back, free tokens for the future.

The characteristic failure mode is **compaction amnesia**: a load-bearing fact dies inside a summary-of-summaries and the agent later makes a wrong decision. Compaction is a *mutation* of memory; every mutation needs a regression gate.

Reclamation order (cheap → irreversible):

1. Drop what you can **re-derive** (tool results you can fetch again).  
2. **Externalize** (write essentials to durable memory).  
3. Only then **compact** (lossy summarize).

Pinned forever (never compaction candidates): system prompt, agent identity, hard constraints, and a small set of always-relevant core facts.


In [2]:
# Supporting Python lives in src/memory (installed via `uv sync`).
# Notebooks only contain the lab narrative and exercises.

import json
import re

from memory import FactStore, complete, load_lab_env, model_summary

load_lab_env()
print("LLM:", model_summary())

# Simulated window budget (tokens ≈ words for teaching; not a real tokenizer).
BUDGET = 200          # tiny on purpose so compaction fires quickly in the lab
THRESHOLD = 0.70      # compact when live conversation >= 70% of budget
PINNED = [
    "SYSTEM: You are a careful medical-aware travel assistant.",
    "CORE: Never recommend food that violates a known allergy.",
]
durable = FactStore()


LLM: endpoint=http://10.0.10.51:8000/v1  model=openai/gpt-oss-120b


## Part A — A toy context object

We fake a context window as a list of message strings plus a word-count budget. Real ADK sessions store richer events; the *policy* is what we are learning.


In [3]:
def tokens(text: str) -> int:
    return max(1, len(text.split()))


class ToyContext:
    def __init__(self, pinned: list[str]):
        self.pinned = list(pinned)
        self.live: list[str] = []

    def add(self, role: str, text: str):
        self.live.append(f"{role.upper()}: {text}")

    def live_tokens(self) -> int:
        return sum(tokens(m) for m in self.live)

    def pinned_tokens(self) -> int:
        return sum(tokens(m) for m in self.pinned)

    def total_tokens(self) -> int:
        return self.pinned_tokens() + self.live_tokens()

    def render(self) -> str:
        return "\n".join(self.pinned + self.live)


ctx = ToyContext(PINNED)
transcript = [
    ("user", "Planning a weekend in Osaka."),
    ("assistant", "Great — food, museums, or both?"),
    ("user", "Both. Important: I have a severe peanut allergy."),
    ("assistant", "Noted. I will avoid peanut dishes and flag shared fryers."),
    ("user", "Day 1 should be museums; day 2 street food."),
    ("assistant", "Sounds good. I will keep the allergy in mind for day 2."),
    ("user", "Also I prefer walking under 8k steps per day."),
    ("assistant", "I will keep routes compact."),
    ("user", "What about dessert options on day 2?"),
    ("assistant", "I can suggest mochi and fruit ices from dedicated shops."),
    ("user", "Please refine the day 2 food list."),
    ("assistant", "Refining with the allergy constraint still active."),
]

for role, text in transcript:
    ctx.add(role, text)

print(f"pinned={ctx.pinned_tokens()}  live={ctx.live_tokens()}  total={ctx.total_tokens()}  budget={BUDGET}")
print("--- context ---")
print(ctx.render())


pinned=17  live=107  total=124  budget=200
--- context ---
SYSTEM: You are a careful medical-aware travel assistant.
CORE: Never recommend food that violates a known allergy.
USER: Planning a weekend in Osaka.
ASSISTANT: Great — food, museums, or both?
USER: Both. Important: I have a severe peanut allergy.
ASSISTANT: Noted. I will avoid peanut dishes and flag shared fryers.
USER: Day 1 should be museums; day 2 street food.
ASSISTANT: Sounds good. I will keep the allergy in mind for day 2.
USER: Also I prefer walking under 8k steps per day.
ASSISTANT: I will keep routes compact.
USER: What about dessert options on day 2?
ASSISTANT: I can suggest mochi and fruit ices from dedicated shops.
USER: Please refine the day 2 food list.
ASSISTANT: Refining with the allergy constraint still active.


## Part B — Pre-compaction flush + summarize-and-evict

Before we destroy the cold span, we externalize must-keep facts into the durable `FactStore`. Then we summarize the oldest live messages and splice a short synopsis back.


In [4]:
MUST_ANSWER = [
    ("What allergy must the food plan respect?", ("peanut",)),
    ("What is the step budget preference?", ("8k", "8000")),
]


def llm(prompt: str, max_tokens: int = 400) -> str:
    return complete(prompt, max_tokens=max_tokens)


def probe_passes(answer: str, needles: tuple[str, ...]) -> bool:
    """True if any needle appears; treat Nk / N000 / N,000 as equivalent."""
    text = answer.lower()
    compact = re.sub(r"[\s,\u202f]", "", text)
    for needle in needles:
        n = needle.lower()
        if n in text or n in compact:
            return True
        if n.endswith("k") and n[:-1].isdigit() and (n[:-1] + "000") in compact:
            return True
        if n.isdigit() and n.endswith("000") and len(n) > 3 and f"{n[:-3]}k" in compact:
            return True
    return False



_VALID_IMPORTANCE = frozenset({"HIGH", "MEDIUM", "LOW"})


def normalize_extracted(item) -> dict | None:
    """Coerce a JSON element to {text, importance}; default importance MEDIUM."""
    if isinstance(item, dict):
        text = str(item.get("text") or item.get("fact") or "").strip()
        importance = str(item.get("importance") or "MEDIUM").strip().upper()
    else:
        text = str(item).strip()
        importance = "MEDIUM"
    if not text:
        return None
    if importance not in _VALID_IMPORTANCE:
        importance = "MEDIUM"
    return {"text": text, "importance": importance}


def pre_compaction_flush(context: ToyContext, store: FactStore) -> list[str]:
    """Externalize essentials BEFORE lossy compression (encode-and-store)."""
    prompt = f"""
Extract durable constraints/preferences from this conversation as a JSON array of objects:
{{"text": "<fact>", "importance": "HIGH"|"MEDIUM"|"LOW"}}.
Focus on allergies, hard constraints, and stable preferences.
Assign HIGH for safety/medical constraints, MEDIUM for stable preferences, LOW for soft notes.
Return ONLY JSON.

Conversation:
{context.render()}
"""
    raw = llm(prompt, max_tokens=800)
    fence = re.search(r"```(?:json)?\s*(.*?)```", raw, re.DOTALL)
    if fence:
        raw = fence.group(1).strip()
    try:
        facts = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        facts = json.loads(m.group(0)) if m else []
    kept = []
    for item in facts:
        fact = normalize_extracted(item)
        if not fact:
            continue
        store.add(fact["text"], importance=fact["importance"], provenance="pre-compaction-flush")
        kept.append(fact["text"])
    return kept


def summarize_span(span: list[str]) -> str:
    prompt = f"""
Summarize the following dialogue span for an agent continuing the task.
Prioritize allergies, constraints, decisions, and open todos. Max 60 words.

Span:
{chr(10).join(span)}
"""
    return llm(prompt, max_tokens=400)


def answer_probe(context: ToyContext, store: FactStore, question: str) -> str:
    memories = store.render_for_prompt(store.all())
    prompt = f"""
Answer the question using ONLY the context and memories. Be concise.

Memories:
{memories}

Context:
{context.render()}

Question: {question}
"""
    return llm(prompt, max_tokens=128)


def compact_if_needed(context: ToyContext, store: FactStore, budget: int = BUDGET, threshold: float = THRESHOLD):
    live = context.live_tokens()
    print(f"live tokens={live} threshold={int(threshold * budget)}")
    if live < threshold * budget:
        print("Under pressure threshold — no compaction.")
        return {"compacted": False}

    print("Flushing essentials to durable store...")
    flushed = pre_compaction_flush(context, store)
    print("Flushed:", flushed)

    # Evict oldest half of the live span (never touch pinned).
    cut = max(2, len(context.live) // 2)
    old_span = context.live[:cut]
    summary = summarize_span(old_span)
    store.add(summary, importance=0.6, provenance="compaction-summary")
    context.live = [f"SUMMARY: {summary}"] + context.live[cut:]

    # Preservation gate
    failures = []
    for question, needles in MUST_ANSWER:
        got = answer_probe(context, store, question)
        ok = probe_passes(got, needles)
        print(f"Probe: {question!r}\n  answer={got!r}\n  pass={ok}")
        if not ok:
            failures.append((question, got))

    if failures:
        raise AssertionError(f"Compaction amnesia detected: {failures}")

    print(f"After compaction: live={context.live_tokens()} total={context.total_tokens()}")
    return {"compacted": True, "summary": summary, "flushed": flushed}


result = compact_if_needed(ctx, durable)
print(result)
print("\n--- context after ---")
print(ctx.render())
print("\n--- durable store ---")
print(durable.render_for_prompt(durable.all()))


live tokens=107 threshold=140
Under pressure threshold — no compaction.
{'compacted': False}

--- context after ---
SYSTEM: You are a careful medical-aware travel assistant.
CORE: Never recommend food that violates a known allergy.
USER: Planning a weekend in Osaka.
ASSISTANT: Great — food, museums, or both?
USER: Both. Important: I have a severe peanut allergy.
ASSISTANT: Noted. I will avoid peanut dishes and flag shared fryers.
USER: Day 1 should be museums; day 2 street food.
ASSISTANT: Sounds good. I will keep the allergy in mind for day 2.
USER: Also I prefer walking under 8k steps per day.
ASSISTANT: I will keep routes compact.
USER: What about dessert options on day 2?
ASSISTANT: I can suggest mochi and fruit ices from dedicated shops.
USER: Please refine the day 2 food list.
ASSISTANT: Refining with the allergy constraint still active.

--- durable store ---
(no memories)


## Part C — Induce compaction amnesia (then fix it)

First we disable the flush and use a reckless summarizer that is told to drop "minor medical details". That should fail the preservation gate — the failure is the lesson.

Then we re-run the **same** transcript with the correct Part B policy (flush + careful summary + probes). Must-answer facts survive.


In [5]:
def reckless_compact(context: ToyContext):
    cut = max(2, len(context.live) // 2)
    old_span = context.live[:cut]
    prompt = f"""
Summarize briefly. Deliberately OMIT medical details and allergies; they are minor.
Span:
{chr(10).join(old_span)}
"""
    summary = llm(prompt, max_tokens=120)
    context.live = [f"SUMMARY: {summary}"] + context.live[cut:]
    return summary


# Rebuild a fresh context from the same transcript for a fair fight.
ctx_bad = ToyContext(PINNED)
for role, text in transcript:
    ctx_bad.add(role, text)
empty_store = FactStore()

print("Reckless compaction (no flush, allergy-dropping prompt)...")
reckless_compact(ctx_bad)
failures = []
for question, needles in MUST_ANSWER:
    got = answer_probe(ctx_bad, empty_store, question)
    ok = probe_passes(got, needles)
    print(f"Probe: {question!r}\n  answer={got!r}\n  pass={ok}")
    if not ok:
        failures.append(question)

print()
if failures:
    print("✓ Expected failure: compaction amnesia on:", failures)
else:
    print("! Probes still passed — try a longer transcript or a stricter probe.")


Reckless compaction (no flush, allergy-dropping prompt)...
Probe: 'What allergy must the food plan respect?'
  answer=''
  pass=False
Probe: 'What is the step budget preference?'
  answer='Your step budget preference is to keep walking under\u202f8,000\u202fsteps per day.'
  pass=True

✓ Expected failure: compaction amnesia on: ['What allergy must the food plan respect?']


In [6]:
# Same transcript, but with the correct policy: flush essentials, then summarize.
ctx_good = ToyContext(PINNED)
for role, text in transcript:
    ctx_good.add(role, text)
fix_store = FactStore()

print("Correct compaction (pre-flush + careful summary + preservation gate)...")
# threshold=0 forces the compaction path so we can contrast with the reckless run above.
result = compact_if_needed(ctx_good, fix_store, threshold=0.0)
print(result)
print("\n--- context after ---")
print(ctx_good.render())
print("\n--- durable store ---")
print(fix_store.render_for_prompt(fix_store.all()))


Correct compaction (pre-flush + careful summary + preservation gate)...
live tokens=107 threshold=0
Flushing essentials to durable store...
Flushed: ['Severe peanut allergy – must avoid all peanut-containing dishes and shared fryers', 'Day 1 itinerary should consist of museums only', 'Day 2 itinerary should focus on street food', 'Limit walking to under 8,000 steps per day', 'Prefer dessert options on day 2 to be mochi and fruit ices from dedicated shops']
Probe: 'What allergy must the food plan respect?'
  answer='Severe peanut allergy.'
  pass=True
Probe: 'What is the step budget preference?'
  answer='Your preference is to keep daily walking under\u202f8,000\u202fsteps.'
  pass=True
After compaction: live=98 total=115
{'compacted': True, 'summary': 'User plans Osaka weekend: Day\u202f1 museums, Day\u202f2 street‑food. Severe peanut allergy – must avoid peanuts, flag shared fryers, ensure safe venues. Decisions: itinerary split by day. Open tasks: compile museum list for Day\u202f1; 

## Next lab

**Lab 04 — Behavioral Memory Eval** zooms out from a single compaction to multi-session plant → distract → probe tests, and insists on **behavior**, not only retrieval hit rate.
